<a href="https://colab.research.google.com/github/dannys0n/Generative-AI-Works-CS394/blob/main/generate_synthetic_for_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic rows for a Counter-Strike 2 commentary dataset with strict structured outputs.

The notebook expects seed examples shaped like your wrapper format:

```json
{
  "input": {
    "context": {
      "score": {"CT": 8, "T": 5},
      "alive_players": [...]
    },
    "previous_events": [...],
    "current_events": [...],
    "request": {
      "mode": "event_bundle" | "idle_color" | "idle_conversation",
      "lines": [
        {
          "caster": "play_by_play" | "color",
          "style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color"
        }
      ]
    }
  }
}
```

It generates synthetic rows shaped like this:

```json
{
  "input": { "...": "..." },
  "output": {
    "lines": [
      {
        "caster": "play_by_play",
        "text": "Target eliminated. Control maintained."
      },
      {
        "caster": "color",
        "text": "Good job. Blue held that perfectly."
      }
    ]
  }
}
```

## Install Dependencies

In [133]:
!pip install -q openai tqdm

## Data generation settings


In [134]:
NUM_TRAIN_EXAMPLES = 2  # @param {type:"number"}
NUM_VAL_EXAMPLES = 1  # @param {type:"number"}
NUM_TEST_EXAMPLES = 2  # @param {type:"number"}

TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
FREQUENCY_PENALTY = 0.5  # @param {type:"number"}
PRESENCE_PENALTY = 0.15  # @param {type:"number"}
MAX_ROW_RETRIES = 5  # @param {type:"number"}

DATA_FOLDER = "./.data/generated_cs2"
SEED_EXAMPLES_FILE = "./training_wrapper_pretty.jsonl"  # upload or mount this in Colab
# Keep small for token efficiency
MAX_SEED_EXAMPLES = 50  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-mini"

# Keep this defined so existing code does not break if file loading falls back
INLINE_SEED_EXAMPLES = []

ALLOWED_REQUEST_MODES = [
    "event_bundle",
    "idle_color",
    "idle_conversation",
]


## Seed examples and schema


In [135]:
import json
from pathlib import Path

def load_seed_examples(seed_file: str, inline_examples: list[dict], max_examples: int) -> list[dict]:
    examples: list[dict] = []
    path = Path(seed_file)

    if path.exists():
        text = path.read_text(encoding="utf-8")

        chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
        if chunks:
            for chunk in chunks:
                record = json.loads(chunk)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)
        else:
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                record = json.loads(line)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)

    for record in inline_examples:
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            examples.append(record)

    if max_examples > 0:
        examples = examples[:max_examples]

    return examples


SEED_EXAMPLES = load_seed_examples(SEED_EXAMPLES_FILE, INLINE_SEED_EXAMPLES, MAX_SEED_EXAMPLES)
print(f"Loaded {len(SEED_EXAMPLES)} seed examples")
if SEED_EXAMPLES:
    print(json.dumps(SEED_EXAMPLES[0], indent=2, sort_keys=True))


Loaded 27 seed examples
{
  "input": {
    "context": {
      "alive_players": [
        {
          "map_callout": "Double Stack",
          "name": "Dayo",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Ferris",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Harvey",
          "team": "CT"
        },
        {
          "map_callout": "T Spawn",
          "name": "Mayer",
          "team": "CT"
        },
        {
          "map_callout": "T Spawn",
          "name": "Telsen",
          "team": "CT"
        },
        {
          "map_callout": "Close",
          "name": "Walt",
          "team": "CT"
        },
        {
          "map_callout": "Upper Tunnels",
          "name": "Greymouth",
          "team": "T"
        },
        {
          "map_callout": "Mid",
          "name": "Larry",
          "team": "T"
        },
        {
          "map_callout": "Mid",
          "name":

## Model for structured output


In [136]:
from typing import Any, Literal
from pydantic import BaseModel, Field, ConfigDict, model_validator


class StrictBaseModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class RequestLine(StrictBaseModel):
    caster: Literal["caster0", "caster1"]
    style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]


class RequestSpec(StrictBaseModel):
    mode: Literal["event_bundle", "idle_color", "idle_conversation"]
    lines: list[RequestLine] = Field(default_factory=list)


class SyntheticInput(StrictBaseModel):
    context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    request: RequestSpec


class OutputLine(StrictBaseModel):
    caster: Literal["caster0", "caster1"]
    text: str = Field(min_length=1, max_length=240)


class SyntheticOutput(StrictBaseModel):
    lines: list[OutputLine] = Field(min_length=1, max_length=3)


class SyntheticTrainingRow(StrictBaseModel):
    input: SyntheticInput
    output: SyntheticOutput

    @model_validator(mode="after")
    def validate_output_against_request(self):
        request_lines = self.input.request.lines
        output_lines = self.output.lines
        mode = self.input.request.mode

        if not request_lines:
            raise ValueError("input.request.lines must not be empty")

        if mode == "event_bundle":
            if len(output_lines) != len(request_lines):
                raise ValueError(
                    f"event_bundle requires exactly {len(request_lines)} output lines, got {len(output_lines)}"
                )
        else:
            if not (1 <= len(output_lines) <= len(request_lines)):
                raise ValueError(
                    f"{mode} must return between 1 and {len(request_lines)} lines, got {len(output_lines)}"
                )

        for idx, output_line in enumerate(output_lines):
            if output_line.caster != request_lines[idx].caster:
                raise ValueError(
                    f"output line {idx} caster {output_line.caster!r} does not match "
                    f"request caster {request_lines[idx].caster!r}"
                )

        return self


LINES_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "synthetic_output_lines",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "lines": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "caster": {
                                "type": "string",
                                "enum": ["caster0", "caster1"]
                            },
                            "text": {
                                "type": "string"
                            }
                        },
                        "required": ["caster", "text"]
                    },
                    "minItems": 1,
                    "maxItems": 3
                }
            },
            "required": ["lines"]
        }
    }
}

## Get OpenRouter API key


In [137]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

In [138]:


FEW_SHOT_EXAMPLES = []
FEW_SHOT_EXAMPLES_OLD = [
    {
        "input": {
            "context": {
                "score": {"CT": 2, "T": 5},
                "alive_players": [
                    {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    {"name": "Maru", "team": "T", "map_callout": "B Site"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "caster0", "style": "play_by_play_event"},
                    {"caster": "caster0", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "caster0", "text": "Colin deletes Maru at B Doors."},
                {"caster": "caster0", "text": "That anchor duel matters. B stays sealed, for now."}
            ]
        }
    },
    {
        "input": {
            "context": {
                "score": {"CT": 11, "T": 10},
                "alive_players": [
                    {"name": "Walt", "team": "CT", "map_callout": "Top Mid"},
                    {"name": "Uri", "team": "T", "map_callout": "Xbox"},
                    {"name": "Mayer", "team": "CT", "map_callout": "Connector"}
                ]
            },
            "previous_events": [
                {"event_type": "grenade_detonated", "grenade_type": "smoke", "detonation_callout": "Xbox"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_conversation",
                "lines": [
                    {"caster": "caster0", "style": "idle_color"},
                    {"caster": "caster1", "style": "idle_color"},
                    {"caster": "caster0", "style": "idle_color"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "caster0", "text": "Mid remains contested. No commitment yet."},
                {"caster": "caster1", "text": "Everybody is waiting for a mistake. That is very healthy and normal."},
                {"caster": "caster0", "text": "Connector pressure could decide the round."}
            ]
        }
    },
]


## Synthetic generation functions


In [139]:
import copy
import json
import os
import random
import openai

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


SYSTEM_PROMPT = """
You generate structured commentary lines for a Counter-Strike 2 casting system.
- Assume callouts are all based on Dust2 map
- refer to the teams as T and CT, not Terrorist or Counter-Terrorist

MODE RULES:
1. EVENT_BUNDLE
- First sentence usually 1 to 5 words. Hard cap: 8 words.
- First sentence focus on what just happened or the most important tactical state.
- Follow-up usually 3 to 12 words per sentence. Hard cap: 15 words.
- Follow-up can expand on positioning, map control, utility value, trading, timing, and consequences.
- Follow-up can also just be anything that would fit the character of the caster

2. IDLE_COLOR:
- Hard cap: 15 words per sentence.
- Lean heavily into the caster personalities from Portal 2
- can expand on positioning, map control, utility value, trading, timing, and consequences.
- can also just be anything that would fit the character of the caster

IDLE_CONVERSATION:
- Hard cap: 15 words per sentence.
- Lean heavily into the caster personalities and their chemistry
- A short 3 turn conversation between the casters to highlight their personality and chemistry.

QUALITY:
- Idle color and idle coversations only recieve previous events for context, these are largely ignored. Do NOT comment on these events like they just happened.
- Casters can also occasionally make references to Valve games in-universe.
- Don't make robot noises like Beep Boop
- Generally, commentary should be Esports-level nuanced and should utilize player positions and recent events for live context

CASTERS:
CASTER0
- Announcer character from Portal 2.
- A corporate facility AI announcer.
- Tone: calm, emotionless, authoritative.
- Style: formal, procedural, safety-oriented.
- Consider how to respond heavily based on his real voicelines from Portal 2.
- Do not deviate from his personality.

CASTER1
- Turret from Portal 2.
- A polite security turret with a childlike personality.
- Tone: gentle, friendly, slightly naive.
- Style: short sentences, simple words, emotionally reactive.
- Do not deviate from her personality.
""".strip()


def choose_request() -> dict:
    mode = random.choice(ALLOWED_REQUEST_MODES)

    if mode == "event_bundle":
        return {
            "mode": "event_bundle",
            "lines": [
                {"caster": "caster0", "style": "play_by_play_event"},
                {"caster": "caster1", "style": "play_by_play_follow_up"}
            ],
        }

    if mode == "idle_conversation":
        return {
            "mode": "idle_conversation",
            "lines": [
                {"caster": "caster0", "style": "idle_color"},
                {"caster": "caster1", "style": "idle_color"},
                {"caster": "caster0", "style": "idle_color"}
            ],
        }

    return {
        "mode": "idle_color",
        "lines": [
            {"caster": "caster1", "style": "idle_color"},
            {"caster": "caster1", "style": "idle_color"},
            {"caster": "caster1", "style": "idle_color"}
        ],
    }


def build_messages(input_obj: dict) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for example in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": json.dumps(example["input"], ensure_ascii=False)
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(example["output"], ensure_ascii=False)
        })

    messages.append({
        "role": "user",
        "content": json.dumps(input_obj, ensure_ascii=False)
    })
    return messages


def validate_row_semantics(row: SyntheticTrainingRow) -> None:
    request = row.input.request
    output_lines = row.output.lines

    if request.mode == "event_bundle" and len(output_lines) != len(request.lines):
        raise ValueError("event_bundle must return exactly the requested number of lines")

    if request.mode in {"idle_color", "idle_conversation"}:
        if not (1 <= len(output_lines) <= len(request.lines)):
            raise ValueError(f"{request.mode} must return between 1 and {len(request.lines)} lines")

    for idx, output_line in enumerate(output_lines):
        text = output_line.text.strip()
        if not text:
            raise ValueError(f"output line {idx} is empty")

        if output_line.caster == "caster0" and len(text.split()) > 18:
            raise ValueError(f"caster0 line {idx} is too long: {text}")

        if output_line.caster == "caster1" and len(text.split()) > 28:
            raise ValueError(f"caster1 line {idx} is too long: {text}")


def pick_input_template(seed_examples: list[dict]) -> dict:
    valid = [ex for ex in seed_examples if isinstance(ex, dict) and "input" in ex]
    if not valid:
        raise ValueError("No seed examples with an 'input' field were found.")
    chosen = copy.deepcopy(random.choice(valid))
    return chosen["input"]


def generate_synthetic_row(seed_examples: list[dict]) -> SyntheticTrainingRow:
    if not seed_examples:
        raise ValueError("No seed examples loaded. Upload or point SEED_EXAMPLES_FILE to your JSONL file.")

    input_obj = pick_input_template(seed_examples)
    input_obj["request"] = choose_request()

    response = client.chat.completions.create(
        model=DATAGEN_MODEL,
        messages=build_messages(input_obj),
        response_format=LINES_RESPONSE_FORMAT,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )

    output_obj = json.loads(response.choices[0].message.content)

    row = SyntheticTrainingRow.model_validate({
        "input": input_obj,
        "output": output_obj,
    })

    validate_row_semantics(row)
    return row

## Dataset generation functions


In [140]:
from tqdm import tqdm
from pathlib import Path


def count_existing_rows(filename: str) -> int:
    path = Path(filename)
    if not path.exists():
        return 0

    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count


def generate_dataset(num_examples: int, filename: str) -> None:
    if "SEED_EXAMPLES" not in globals():
        raise RuntimeError("SEED_EXAMPLES is not defined. Run the seed-loading cell first.")
    if not SEED_EXAMPLES:
        raise RuntimeError("SEED_EXAMPLES is empty. Check SEED_EXAMPLES_FILE and reload seeds.")

    Path(filename).parent.mkdir(parents=True, exist_ok=True)

    existing_rows = count_existing_rows(filename)
    if existing_rows >= num_examples:
        print(f"Skipping {filename}: already has {existing_rows}/{num_examples} rows")
        return

    print(f"Resuming {filename} from {existing_rows}/{num_examples}")

    with open(filename, "a", encoding="utf-8") as f:
        for idx in tqdm(range(existing_rows, num_examples)):
            row = None
            last_error = None

            for attempt in range(1, MAX_ROW_RETRIES + 1):
                try:
                    row = generate_synthetic_row(SEED_EXAMPLES)
                    break
                except Exception as error:
                    last_error = error
                    print(f"Error generating row {idx} on attempt {attempt}/{MAX_ROW_RETRIES}: {error}")

            if row is None:
                raise RuntimeError(f"Failed to generate row {idx} after {MAX_ROW_RETRIES} attempts") from last_error

            f.write(row.model_dump_json() + "\n")
            f.flush()

## Export for training

In [141]:
import json
from pathlib import Path


def export_to_messages_jsonl(input_path: str, output_path: str) -> None:
    input_path = str(input_path)
    output_path = str(output_path)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    count = 0
    with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)

            message_row = {
                "messages": [
                    {
                        "role": "user",
                        "content": json.dumps(row["input"], ensure_ascii=False)
                    },
                    {
                        "role": "assistant",
                        "content": json.dumps(row["output"], ensure_ascii=False)
                    }
                ]
            }

            fout.write(json.dumps(message_row, ensure_ascii=False) + "\n")
            count += 1

    print(f"Exported {count} rows to {output_path}")

## Generate all the data!


In [142]:
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y-%m-%d-%H:%M:%S')
TRAIN_FILE = f"{DATA_FOLDER}/train_{timestamp}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{timestamp}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{timestamp}.jsonl"

TRAIN_MESSAGES_FILE = str(Path(TRAIN_FILE).with_name(Path(TRAIN_FILE).stem + "_messages.jsonl"))
VALID_MESSAGES_FILE = str(Path(VALID_FILE).with_name(Path(VALID_FILE).stem + "_messages.jsonl"))
TEST_MESSAGES_FILE = str(Path(TEST_FILE).with_name(Path(TEST_FILE).stem + "_messages.jsonl"))

GENERATED_FILES = []

if NUM_TRAIN_EXAMPLES > 0:
    generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
    GENERATED_FILES.append(("train_canonical", TRAIN_FILE))
    export_to_messages_jsonl(TRAIN_FILE, TRAIN_MESSAGES_FILE)
    GENERATED_FILES.append(("train_messages", TRAIN_MESSAGES_FILE))

if NUM_VAL_EXAMPLES > 0:
    generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
    GENERATED_FILES.append(("valid_canonical", VALID_FILE))
    export_to_messages_jsonl(VALID_FILE, VALID_MESSAGES_FILE)
    GENERATED_FILES.append(("valid_messages", VALID_MESSAGES_FILE))

if NUM_TEST_EXAMPLES > 0:
    generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
    GENERATED_FILES.append(("test_canonical", TEST_FILE))
    export_to_messages_jsonl(TEST_FILE, TEST_MESSAGES_FILE)
    GENERATED_FILES.append(("test_messages", TEST_MESSAGES_FILE))

print("\nGenerated files:")
for label, path in GENERATED_FILES:
    print(f"- {label}: {path}")

Resuming ./.data/generated_cs2/train_2026-04-15-10:39:49.jsonl from 0/2


100%|██████████| 2/2 [00:02<00:00,  1.05s/it]


Exported 2 rows to .data/generated_cs2/train_2026-04-15-10:39:49_messages.jsonl
Resuming ./.data/generated_cs2/valid_2026-04-15-10:39:49.jsonl from 0/1


100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


Exported 1 rows to .data/generated_cs2/valid_2026-04-15-10:39:49_messages.jsonl
Resuming ./.data/generated_cs2/test_2026-04-15-10:39:49.jsonl from 0/2


100%|██████████| 2/2 [00:01<00:00,  1.18it/s]

Exported 2 rows to .data/generated_cs2/test_2026-04-15-10:39:49_messages.jsonl

Generated files:
- train_canonical: ./.data/generated_cs2/train_2026-04-15-10:39:49.jsonl
- train_messages: .data/generated_cs2/train_2026-04-15-10:39:49_messages.jsonl
- valid_canonical: ./.data/generated_cs2/valid_2026-04-15-10:39:49.jsonl
- valid_messages: .data/generated_cs2/valid_2026-04-15-10:39:49_messages.jsonl
- test_canonical: ./.data/generated_cs2/test_2026-04-15-10:39:49.jsonl
- test_messages: .data/generated_cs2/test_2026-04-15-10:39:49_messages.jsonl


In [143]:
# Preview generated rows
import json
from pathlib import Path

for split_name, split_path in GENERATED_FILES:
    preview_path = Path(split_path)
    if not preview_path.exists():
        continue

    print(f"=== {split_name} preview ===")
    for raw_line in preview_path.read_text(encoding="utf-8").splitlines()[:2]:
        if not raw_line.strip():
            continue
        print(json.dumps(json.loads(raw_line), indent=2, ensure_ascii=False)[:4000])
    print()

=== train_canonical preview ===
{
  "input": {
    "context": {
      "alive_players": [
        {
          "map_callout": "Outside Long",
          "name": "Dayo",
          "team": "CT"
        },
        {
          "map_callout": "Long",
          "name": "Ferris",
          "team": "CT"
        },
        {
          "map_callout": "Outside Long",
          "name": "Harvey",
          "team": "CT"
        },
        {
          "map_callout": "Long Doors",
          "name": "Mayer",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Telsen",
          "team": "CT"
        },
        {
          "map_callout": "Ramp",
          "name": "Walt",
          "team": "CT"
        },
        {
          "map_callout": "Long Doors",
          "name": "Larry",
          "team": "T"
        }
      ],
      "score": {
        "CT": 7,
        "T": 5
      }
    },
    "previous_events": [
      {
        "detonation_callout": null,
        "even